# Notebook 3 – Spatial Profiles and 2D Spectra

This notebook explores the **spatial structure** of the cantilever motion — how the
deflection and slope vary along the beam length, and how this variation changes with
drive frequency and contact load.

## What you will learn

1. How to solve the BVP at a **single frequency** and inspect the resulting displacement
   profile z(x) and slope profile z'(x) along the cantilever
2. How to compare mode shapes **below, at, and above resonance** to understand the
   phase and amplitude evolution through the resonance
3. How the mode shape evolves as **contact stiffness k₁ increases** (from free-beam
   shape toward a clamped-pinned geometry)
4. How to compute a full **2D spatial-frequency spectrogram** z(x, f) as a heatmap —
   the complete picture of cantilever dynamics

## Background

The Euler–Bernoulli model returns the full complex mode shape z(x) at each frequency,
not just the scalar tip amplitude |z'(L1)|. This spatial information is useful for:
- Identifying nodes (zero-crossing positions) of higher eigenmodes
- Understanding how the drive geometry (distributed vs tip) excites each mode
- Designing experiments that maximise tip-sample force sensitivity

**Probe:** Budget Sensors Multi 75-G  (k = 2.8 N/m, f_free ≈ 75 kHz)

All figures are saved to `../results/`.

If `cant.tip_offset > 0`, the dynamic domain ends at the contact point `L1 = cant.contact_x`, and the x-normalized plots in this notebook use that contact position.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../src'))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from eb_cr_afm import (
    default_params_multi75,
    ContactParams, SolverOptions,
    solve_single_frequency, frequency_sweep, sho_analysis,
    compute_2d_spectra,
    plot_displacement_profile_plotly,
    plot_modes_and_spectra,
)

# Results directory for saving figures
RESULTS_DIR = os.path.join(os.getcwd(), '../results')
os.makedirs(RESULTS_DIR, exist_ok=True)

cant, contact, exc, opts = default_params_multi75()
print('Ready. Cantilever L =', cant.L*1e6, 'µm')
print(f'Results will be saved to: {os.path.abspath(RESULTS_DIR)}')

## 1 – Find the contact resonance frequency

Before solving for mode shapes we run a coarse frequency sweep to locate the dominant
contact resonance. We use `k₁ = 30 × k_cant = 84 N/m`, which places the resonance
in the 200–700 kHz range — a moderate-contact regime where mode shape evolution is
visually interesting.

The coarse sweep uses `f_step=2000 Hz` for speed. Once f_res is identified, we
solve at three specific frequencies (below / at / above) with a finer grid to get
accurate spatial profiles.

In [ ]:
# Moderate contact stiffness: k1 = 5*k ≈ 14 N/m
contact_mod = ContactParams(k1=30*cant.k, k2=0.85*5*cant.k, c1=0.0, c2=0.0, psi=0.0)

opts_fast = SolverOptions(n_points=150, f_start=50e3, f_stop=850e3, f_step=2000.0, g=20.0)
sweep = frequency_sweep(cant, contact_mod, exc, opts_fast, verbose=False)
sho  = sho_analysis(sweep)[0]

f_res = sho['f_res']
print(f'Contact resonance: {f_res/1e3:.1f} kHz  (Q = {sho["Q"]:.0f})')

In [ ]:
freqs_to_solve = {
    'Below resonance': max(f_res - 20e3, 55e3),
    'At resonance':    f_res,
    'Above resonance': f_res + 20e3,
}

opts_fine = SolverOptions(n_points=300, f_start=50e3, f_stop=400e3, f_step=500.0, g=20.0)
results = {}
for label, freq in freqs_to_solve.items():
    results[label] = solve_single_frequency(freq, cant, contact_mod, exc, opts_fine)
    print(f'  {label}: f = {freq/1e3:.1f} kHz, A = {results[label]["A"]*1e12:.4f} pm/rad')

## 2 – Solve at three key frequencies

We evaluate the mode shape at three characteristic points:
- **Below resonance:** The cantilever responds quasi-statically, with amplitude
  well below the peak and phase near 0°
- **At resonance:** Maximum amplitude, phase = −90° (complex tip slope is purely imaginary)
- **Above resonance:** The drive is faster than the natural frequency, amplitude
  falls again, and phase approaches −180°

The amplitude ratio At/Abelow ≈ At/Aabove ≈ Q confirms the Q-factor from the
SHO fit. Note that the profile shapes change significantly: at resonance the mode
shape peaks near the tip, while off-resonance the profile is more beam-like.

## 3 – Displacement profile at resonance (interactive)

`plot_displacement_profile_plotly()` shows the full complex mode shape z(x) and
slope z'(x) at a single frequency. The shaded grey area is the amplitude envelope |z|
or |z'|, while the solid/dashed lines show Re(z) and Im(z).

Physical interpretation:
- **z(x):** Transverse displacement of the beam at position x. The clamped boundary
  condition forces z(0) = z'(0) = 0. The tip is at x = L1 = cant.contact_x.
- **z'(x) = dz/dx:** Local slope angle. The optical lever detects z'(L1) at the contact point.
  Bending moment M(x) = EI × z''(x), shear force Q(x) = −EI × z'''(x).

The vertical dotted line marks the tip/contact position x = L1 = cant.contact_x and shows the tip amplitude A = |z'(L1)|.

In [ ]:
fig = plot_displacement_profile_plotly(results['At resonance'],
                                       title='Cantilever Profile at Contact Resonance')
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb03_profile_at_resonance.html'))
print('Saved: nb03_profile_at_resonance.html')

## 4 – Overlay: |z(x)| at below / at / above resonance

All three profiles are **normalised** by the maximum amplitude at the tip |z(L1)|
so that the shape (not the scale) can be compared.

Key observations:
- At resonance, the normalised profile has a clear broad maximum near the tip
- Below and above resonance the profile is compressed and more uniform
- The mode shape does not change sign at these three frequencies (all within the
  first resonance lobe); sign changes only appear between resonances on a log-scale sweep

In [ ]:
colors = {'Below resonance': '#aec7e8', 'At resonance': '#1f77b4', 'Above resonance': '#98df8a'}

fig = go.Figure()
for label, res in results.items():
    x_um = res['x'] * 1e6
    # Normalize to tip amplitude
    z_norm = np.abs(res['z']) / max(np.abs(res['z'][-1]), 1e-30)
    fig.add_trace(go.Scatter(
        x=x_um, y=z_norm,
        mode='lines', name=label,
        line=dict(color=colors[label], width=2.5),
        hovertemplate='x = %{x:.1f} µm<br>|z|/|z(L1)| = %{y:.3f}<extra></extra>',
    ))

fig.update_xaxes(title_text='Position x (µm)')
fig.update_yaxes(title_text='Normalized |z(x)| / |z(L1)|')
fig.update_layout(
    title='Cantilever shape at different frequencies',
    height=380, template='plotly_white', hovermode='x unified',
)
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb03_profiles_below_at_above.html'))
print('Saved: nb03_profiles_below_at_above.html')

## 5 – Mode shape evolution with contact load (fixed frequency)

Here we fix the excitation frequency and vary k₁ to see how the mode shape changes
as the tip–sample contact stiffens.

**Physical picture:**
- At k₁ → 0 (free cantilever): mode shape is the first free-beam eigenmode —
  a smooth arch peaking near the tip, with the beam acting as a clamped-free beam
- At k₁ → ∞ (rigid contact): mode shape approaches the clamped-pinned shape —
  a node appears at the tip position, and the profile has an inflection point
  that migrates toward the fixed base

The intermediate regime (k₁ ≈ k_cant to 100 × k_cant) is most relevant for
typical AFM experiments. The colour scale (viridis) maps from low k₁ (blue/purple)
to high k₁ (yellow/green).

**Note:** At the fixed probe frequency `f_probe = 10 kHz`, the cantilever is
far below any resonance, so the response is quasi-static and the profiles are
smooth and monotonic.

In [ ]:
import plotly.colors as pc

# ── User settings ──────────────────────────────────────────────────────────
f_probe   = 10e3           # Hz — probe frequency (far below resonance → quasi-static)
k1_values = np.logspace(   # N/m — contact stiffness range to sweep
    np.log10(0.1 * cant.k),
    np.log10(10000 * cant.k),
    5,
)
k2_ratio  = 0.85           # k2 / k1 (isotropic surface limit)
# ───────────────────────────────────────────────────────────────────────────

opts_prof = SolverOptions(n_points=300, f_start=50e3, f_stop=100e3,
                          f_step=500.0, g=20.0, method='analytical')

print(f'Solving {len(k1_values)} profiles at f = {f_probe/1e3:.1f} kHz ...')
profiles = []
for k1 in k1_values:
    c = ContactParams(k1=k1, k2=k2_ratio*k1, c1=0.0, c2=0.0, psi=0.0)
    r = solve_single_frequency(f_probe, cant, c, exc, opts_prof)
    profiles.append({'k1': k1, 'x': r['x'], 'z': r['z'], 'zp': r['zp']})
print('Done.')

# ── Build colour scale (log k1 → colour) ──────────────────────────────────
k1_log   = np.log10(k1_values)
k1_log_n = (k1_log - k1_log.min()) / (k1_log.max() - k1_log.min())
colorscale = pc.sequential.Viridis
colors = pc.sample_colorscale(colorscale, k1_log_n)

# ── Plot |z(x)| and |z'(x)|, each normalised to its own peak ──────────────
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=(
        f"|z(x)|  —  f = {f_probe/1e3:.1f} kHz, k₁ swept",
        "|z'(x)|  (slope)",
    ),
)

for p, color in zip(profiles, colors):
    x_um    = p['x'] * 1e6
    label   = f"k₁ = {p['k1']:.1f} N/m"
    z_norm  = np.abs(p['z'])  / (np.max(np.abs(p['z']))  or 1.0)
    zp_norm = np.abs(p['zp']) / (np.max(np.abs(p['zp'])) or 1.0)
    fig.add_trace(
        go.Scatter(x=x_um, y=z_norm,
                   name=label, legendgroup=label,
                   line=dict(color=color, width=2.5),
                   hovertemplate='x = %{x:.1f} µm<br>|z| (norm.) = %{y:.3f}<extra>' + label + '</extra>'),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=x_um, y=zp_norm,
                   name=label, legendgroup=label, showlegend=False,
                   line=dict(color=color, width=2.5),
                   hovertemplate="x = %{x:.1f} µm<br>|z'| (norm.) = %{y:.3f}<extra>" + label + '</extra>'),
        row=2, col=1,
    )

# Colourbar via dummy scatter
fig.add_trace(
    go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(
            colorscale=colorscale,
            cmin=k1_values.min(), cmax=k1_values.max(),
            color=[0], colorbar=dict(
                title='k₁ (N/m)', tickformat='.0f',
                len=0.9, x=1.02,
            ),
            showscale=True,
        ),
        showlegend=False,
    ),
    row=1, col=1,
)

fig.update_xaxes(title_text='Position x (µm)', row=2, col=1)
fig.update_yaxes(title_text='|z|  (norm.)',  row=1, col=1)
fig.update_yaxes(title_text="|z'| (norm.)", row=2, col=1)
fig.update_layout(
    height=560, template='plotly_white', hovermode='x unified',
    showlegend=False,
    title=f'Mode shape evolution with contact load  (f = {f_probe/1e3:.1f} kHz)',
)
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb03_mode_shape_vs_k1.html'))
print('Saved: nb03_mode_shape_vs_k1.html')

## 5 – Full 2D spatial-frequency spectra z(x, f)

`compute_2d_spectra()` builds a complete (x, f) map by solving the BVP at every
combination of spatial position and drive frequency. The result is an array of shape
(n_x_points, n_frequencies) for both amplitude and phase of z(x, f).

**What to look for in the heatmap:**
- **Bright vertical bands:** Resonance frequencies where the entire beam responds strongly
- **Spatial nodes:** Low-amplitude regions at positions where a given mode has a node
  (zero crossing). These shift with frequency as different modes are excited.
- **Phase jumps:** The right panel shows 90° phase transitions at each resonance,
  visible as sharp colour changes across the frequency axis

**Computational cost:** Each cell in the (x, f) grid requires a separate BVP solve.
With `n_x_points=50` and `f_step=1000 Hz` over 50–1200 kHz, this is ~57,500 solves.
The analytical solver makes this feasible; the BVP solver would be ~30× slower.

In [ ]:
opts_2d = SolverOptions(n_points=100, f_start=50e3, f_stop=1200e3, f_step=1000.0, g=20.0)

spectra = compute_2d_spectra(
    cant, contact_mod, exc, opts_2d,
    n_x_points=50,
    verbose=True,
)
print('2D spectra done.')

In [ ]:
freqs_kHz = spectra['freqs'] / 1e3
x_um      = spectra['x_grid'] * 1e6
z_amp_log = np.log10(np.clip(spectra['z_amp'], 1e-17, None))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('|z(x,f)| amplitude (log₁₀ m)', 'z(x,f) phase (°)'),
                    horizontal_spacing=0.15)

fig.add_trace(
    go.Heatmap(x=x_um, y=freqs_kHz, z=z_amp_log.T,
               colorscale='RdBu_r',
               colorbar=dict(title='log₁₀(m)', x=0.44, len=0.85),
               hovertemplate='x = %{x:.1f} µm<br>f = %{y:.1f} kHz<br>log A = %{z:.2f}<extra></extra>'),
    row=1, col=1)

fig.add_trace(
    go.Heatmap(x=x_um, y=freqs_kHz, z=spectra['z_phase'].T,
               colorscale='Plasma', zmin=-180, zmax=180,
               colorbar=dict(title='Phase (°)', x=1.01, len=0.85),
               hovertemplate='x = %{x:.1f} µm<br>f = %{y:.1f} kHz<br>φ = %{z:.1f}°<extra></extra>'),
    row=1, col=2)

for col in (1, 2):
    fig.update_xaxes(title_text='Position x (µm)', row=1, col=col)
    fig.update_yaxes(title_text='Frequency (kHz)', row=1, col=col)

fig.update_layout(height=480, template='plotly_white',
                  title='2D Spatial-Frequency Spectra  z(x, f)')
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb03_2d_spatial_frequency_spectra.html'))
print('Saved: nb03_2d_spatial_frequency_spectra.html')

## 6 – Mode shapes and transfer functions across k₁

`plot_modes_and_spectra()` is a comprehensive four-panel matplotlib summary that
combines spatial mode shapes with the full transfer function for each k₁ value.

### Panel layout

| Panel | x-axis | y-axis | Content |
|-------|--------|--------|---------|
| (a) | x/L | \|z\| (norm.) | Displacement mode shape at selected resonance |
| (b) | x/L | \|z'\| (norm.) | Slope mode shape at selected resonance |
| (c) | Frequency (MHz) | \|z'(L1)\| | Transfer-function amplitude (log y-axis) |
| (d) | Frequency (MHz) | Phase (°) | Transfer-function phase |

Curves are colour-coded **blue → red** (plasma) from lowest to highest k₁.
A **●** marker on panels (c)/(d) shows which resonance peak was used for the
mode-shape solve — useful for spotting peak mis-detections.

### Solver choice

The **analytical** solver is used here because the Multi 75-G probe has no
distributed electrostatic load (`q(x) = 0`). The analytical particular solution
is identically zero, making the solve exact and ~10–50× faster than the BVP solver.

In [ ]:
# ── Solver options ──────────────────────────────────────────────────────────
# Use the analytical method (valid for Multi 75-G because q(x) = 0):
# no distributed electrostatic load → exact particular solution,
# ~10–50× faster than the numerical BVP solver.
opts_analytical = SolverOptions(
    g=opts.g,
    method='analytical',
    n_points=opts.n_points,
    f_start=opts.f_start,
    f_stop=opts.f_stop,
    f_step=opts.f_step,
)

# ── Contact stiffness range ──────────────────────────────────────────────────
# 8 log-spaced values spanning free-cantilever (k₁ → 0) to stiff Si-like contact.
k1_sweep = np.logspace(
    np.log10(cant.k / 2),       # ~1.4 N/m  — near free-cantilever limit
    np.log10(5000 * cant.k),    # ~5600 N/m — stiff sample
    8,
)
print(f'k₁ range: {k1_sweep[0]:.2f} – {k1_sweep[-1]:.0f} N/m  ({len(k1_sweep)} values)')

### 6a – First eigenmode (mode_index=0)

The **first contact resonance** shifts from the free-cantilever frequency
(~75 kHz, k₁ → 0) toward the clamped-pinned limit (~329 kHz, k₁ → ∞).
The displacement profile evolves from a single-arch free-beam shape toward
a shape with a node forming near the tip, and the slope profile shows
increasing tip sensitivity as the contact stiffens.

In [ ]:
# First eigenmode: mode_index=0 selects the lowest-frequency peak in the
# 0–1 MHz sweep for each k₁.  The ● markers on panels (c)/(d) confirm
# which peak was chosen — useful for spotting mis-detections.
fig_mode1 = plot_modes_and_spectra(
    cant, contact, exc,
    opts=opts_analytical,
    k1_values=k1_sweep,
    k2_over_k1=0.0,
    mode_index=0,          # 0 = first (lowest-frequency) eigenmode
    f_step=1000.0,         # 1 kHz step → 1000 points over 0–1 MHz
)
fig_mode1.suptitle('Multi 75-G — First contact-resonance eigenmode', y=1.01)
fig_mode1.show()

# Save figure
fig_mode1.savefig(os.path.join(RESULTS_DIR, 'nb03_mode1_shapes_and_spectra.png'),
                  dpi=150, bbox_inches='tight')
print('Saved: nb03_mode1_shapes_and_spectra.png')

### 6b – Second eigenmode (mode_index=1)

The **second contact resonance** starts near the free-cantilever second harmonic
(~470 kHz for Multi 75-G) and shifts upward with k₁. Its displacement profile
has **two arches** (one node along the beam), and the slope profile shows a more
complex shape — important for higher-mode CR-AFM techniques such as dual-frequency
or bimodal imaging.

The second mode is more sensitive to lateral stiffness (k₂) because its inflection
point — where bending moment changes sign — can be positioned near the tip for
certain k₁ values, making it a selective probe of lateral contact properties.

If only one peak is detected for some k₁ values (e.g. the second mode falls
outside the sweep window), the code falls back to mode 0 and prints a warning.

In [ ]:
# Second eigenmode: mode_index=1 selects the second-lowest-frequency peak
# in the 0–1 MHz sweep for each k₁.
fig_mode2 = plot_modes_and_spectra(
    cant, contact, exc,
    opts=opts_analytical,
    k1_values=k1_sweep,
    k2_over_k1=0.0,
    mode_index=1,          # 1 = second (next-lowest-frequency) eigenmode
    f_step=1000.0,         # 1 kHz step → 1000 points over 0–1 MHz
)
fig_mode2.suptitle('Multi 75-G — Second contact-resonance eigenmode', y=1.01)
fig_mode2.show()

# Save figure
fig_mode2.savefig(os.path.join(RESULTS_DIR, 'nb03_mode2_shapes_and_spectra.png'),
                  dpi=150, bbox_inches='tight')
print('Saved: nb03_mode2_shapes_and_spectra.png')